# 03 · 标准训练循环(★★★★★)

把 02 放进 epoch/batch 循环,加:AdamW/FusedAdam、bf16 autocast、warmup+cosine、梯度裁剪、验证、**存最优权重**、日志。

> 完整可跑版见仓库根同级 `fast_distill3.py`(int8 缓存加速,教师前向只跑1次)。本 notebook 讲**循环骨架**,用少量样本演示。

In [ ]:
# ============ 公共设置(每个 notebook 先跑这一格)============
import os, sys, json, glob, math, time, numpy as np, torch, torch.nn.functional as F
import warnings; warnings.filterwarnings("ignore")

BASE = "/public/home/xdzs2026_c296"          # ★你的主目录,若不同改这里
BASELINE = f"{BASE}/xiandao2026-AI4S/pangu_weather"   # 官方 baseline(含 maxvit3d_student.py, conf, data)
CKPT = f"{BASELINE}/data/checkpoints/model_bak.pth"   # 教师权重
TRAIN_DATA = f"{BASE}/era5_real"              # 训练数据(13年)
VAL_DATA   = f"{BASE}/era5_testc"             # 验证/测试数据(2000年)
WORK = f"{BASE}/_learn_work"                  # 本教程的工作目录(存中间文件)
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, BASELINE)                  # 为了 import maxvit3d_student
print("torch", torch.__version__, "| DCU 可用:", torch.cuda.is_available())


In [ ]:
from onescience.models.pangu import Pangu
from maxvit3d_student import MaxVit3DStudent
import maxvit3d_student as M; M.set_sdpa(True)
dev = 0
teacher = Pangu(img_size=(721,1440)).to(dev).eval()
teacher.load_state_dict(torch.load(CKPT, map_location=f"cuda:{dev}", weights_only=False)["model_state_dict"])
for p in teacher.parameters(): p.requires_grad_(False)
student = MaxVit3DStudent(patch_size=(2,16,16), embed_dim=96, depths=(2,4,2),
                          num_heads=(6,12,6), mlp_ratio=2.0).to(dev)

## 1) 优化器 + warmup+cosine 学习率 + 损失
容器里也可用 `from apex.optimizers import FusedAdam`(更快),这里用标准 AdamW。

In [ ]:
EPOCHS, WARMUP, LR = 5, 1, 6e-4          # 演示用 5 轮;实战 30
opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.05, betas=(0.9,0.95))
def lr_at(ep):
    if ep < WARMUP: return (ep+1)/WARMUP
    t = (ep-WARMUP)/max(1, EPOCHS-WARMUP); return 0.5*(1+math.cos(math.pi*t))
sw = torch.tensor([1.5,0.77,0.66,3.0], device=dev).view(1,4,1,1); pw = torch.ones(1,65,1,1, device=dev)
def wl1(a,b,w,lw): return lw*(F.l1_loss(a,b,reduction="none")*w).mean()

## 2) 准备几个训练样本(演示:随机 3 个;实战用 01 的 x72 从真实数据读)
> 真实训练强烈建议**预处理缓存**(教师输出只算一次),否则每轮重算教师很慢。见 `fast_distill3.py`。

In [ ]:
N = 3
xs = [torch.randn(1,72,721,1440, device=dev) for _ in range(N)]   # 占位;实战替换成真实 x72
# 预算教师输出(冻结,常量,缓存起来)
touts = []
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    for x in xs:
        ts_, tu_ = teacher(x); touts.append((ts_.float(), tu_.reshape(1,65,721,1440).float()))
print("教师输出已缓存", len(touts), "个")

## 3) ★训练循环骨架(train/eval、前向、反向、裁剪、step、scheduler、存最优、日志)

In [ ]:
best = 1e9
for ep in range(EPOCHS):
    for g in opt.param_groups: g["lr"] = LR * lr_at(ep)      # 手动 warmup+cosine
    student.train(); run = 0.0
    for x, (ts_, tu_) in zip(xs, touts):
        with torch.autocast("cuda", dtype=torch.bfloat16):    # bf16 混合精度
            ss, su = student(x); su = su.reshape(1,65,721,1440)
        pred = torch.cat([ss.float(), su.float()], 1)
        loss = wl1(pred[:,:4], ts_, sw, 0.25) + wl1(pred[:,4:], tu_, pw, 1.0)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step(); run += loss.item()
    # —— 验证(演示直接用训练样本;实战用独立 val)——
    student.eval(); v = 0.0
    with torch.no_grad():
        for x, (ts_, tu_) in zip(xs, touts):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                ss, su = student(x); su = su.reshape(1,65,721,1440)
            pred = torch.cat([ss.float(), su.float()], 1)
            v += (wl1(pred[:,:4],ts_,sw,0.25)+wl1(pred[:,4:],tu_,pw,1.0)).item()
    v /= N
    if v < best:                                             # ★只存最优
        best = v
        torch.save({"model_state_dict": student.state_dict(),
                    "config": {"embed":96,"depths":[2,4,2],"heads":[6,12,6],"patch":[2,16,16],"mlp_ratio":2.0},
                    "epoch": ep+1, "best_loss": best}, f"{WORK}/student_best.pth")
    print(f"ep{ep+1}/{EPOCHS} train={run/N:.4f} val={v:.4f} lr={LR*lr_at(ep):.2e}" + ("  [saved best]" if v==best else ""))
print("训练完成,最优权重 ->", f"{WORK}/student_best.pth")

### ✅ 要点:warmup+cosine 手动调 lr、bf16 autocast、clip 1.0、**只存 val 最优**、每轮日志。
实战整套(含 int8 缓存 + Muon/增广/频域可选)见 `fast_distill3.py`;闭卷把这个骨架背下来即可。